In [1]:
import pandas as pd
import numpy as np
import joblib

### Utworzenie wersji light bazy danych (20k artykułów)

# wczytanie pełnych danych
print("Wczytywanie danych...")
df_full = joblib.load('df_with_all_clusters.joblib')
embeddings_full = np.load('embeddings_full.npy')

# sprawdzenie spójności
assert len(df_full) == len(embeddings_full), "Błąd: Liczba wierszy i wektorów się nie zgadza!"
print(f"Załadowano {len(df_full)} dokumentów.")

# losowanie 20000 indeksów
n_samples = 20000
print(f"Losowanie {n_samples} próbek...")

### ziarno losowości dla powtarzalności
np.random.seed(42)
# wybór losowych indeksów z zakresu 0...len(df)
random_indices = np.random.choice(len(df_full), n_samples, replace=False)

# wycinanie danych
### wyciąganie wierszy z dataframe
df_light = df_full.iloc[random_indices].copy()
### wyciąganie wierszy z macierzy numpy
embeddings_light = embeddings_full[random_indices]

# reset indeksu w df_light, aby był 0, 1, 2...
df_light.reset_index(drop=True, inplace=True)

print(f"Nowy kształt DataFrame (wersja light): {df_light.shape}")
print(f"Nowy kształt Embeddings (wersja light): {embeddings_light.shape}")

# zapis wersji Light
joblib.dump(df_light, 'df_light.joblib')
np.save('embeddings_light.npy', embeddings_light)

print("Zapisano pliki w wersji light...")

Wczytywanie danych...
Załadowano 850367 dokumentów.
Losowanie 20000 próbek...
Nowy kształt DataFrame (wersja light): (20000, 15)
Nowy kształt Embeddings (wersja light): (20000, 768)
Zapisano pliki w wersji light...


# Pinecone

In [2]:
import os
import time
import pandas as pd
import numpy as np
import joblib
from pinecone import Pinecone, ServerlessSpec
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

## konfiguracja
INDEX_NAME = "main-projects-index"
NAMESPACE = "covid-papers" ### przestrzeń nazw dla tego projektu
DIMENSION = 768 ### wymiar embeddingów SPECTER

print("Inicjalizacja Pinecone...")
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

## utworzenie Indeksu (Bazy Danych)
### sprawdzenie czy indeks już jest żeby nie tworzyć duplikatów i błędu
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Tworzenie nowego indeksu: {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine", #### najlepsza metryka dla embeddingów tekstowych
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1" #### standardowy region dla Free Tier
        )
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(1)
    print("Indeks utworzony i gotowy.")
else:
    print(f"Indeks {INDEX_NAME} już istnieje. Łączenie...")

## łączenie z indeksem
index = pc.Index(INDEX_NAME)

## wczytanie danych w wersji light - żeby były w pamięci
print("Wczytywanie danych lokalnych...")
df_light = joblib.load('df_light.joblib')
embeddings_light = np.load('embeddings_light.npy')

## wysyłanie danych (Upsert) w partiach (Batching)
##### Pinecone zaleca wysyłanie danych w paczkach po np. 100-200 rekordów
batch_size = 200
total_vectors = len(df_light)

print(f"\nRozpoczynam wysyłanie {total_vectors} wektorów do Pinecone...")

for i in tqdm(range(0, total_vectors, batch_size)):
    ### zakres partii
    i_end = min(i + batch_size, total_vectors)
    
    ### pobieranie ID --> muszą być stringami
    #### indeks z dataframe jako ID
    ids = [str(x) for x in df_light.index[i:i_end]]
    
    ### pobieranie wektorów
    vecs = embeddings_light[i:i_end]
    
    ### pobieranie metadanych (tytuł + abstrakt)
    ##### Pinecone ma limit wielkości metadanych (40KB/wektor) dlatego przycinanie abstraktu dla bezpieczeństwa
    metadatas = []
    for idx in range(i, i_end):
        row = df_light.iloc[idx]
        meta = {
            "title": str(row['title']),
            #### przycinanie do 6000 znaków, żeby nie zapchać metadanych - wystarczy do RAG
            "abstract": str(row['abstract'])[:6000] if pd.notnull(row['abstract']) else ""
        }
        metadatas.append(meta)
    
    ## lista krotek do wysłania: (id, vector, metadata)
    to_upsert = list(zip(ids, vecs, metadatas))
    
    ## wysyłanie do Pinecone
    index.upsert(vectors=to_upsert, namespace=NAMESPACE)

print("\nZakończono! Dane są w chmurze Pinecone.")

Inicjalizacja Pinecone...
Tworzenie nowego indeksu: main-projects-index...
Indeks utworzony i gotowy.
Wczytywanie danych lokalnych...

Rozpoczynam wysyłanie 20000 wektorów do Pinecone...


  0%|          | 0/100 [00:00<?, ?it/s]


Zakończono! Dane są w chmurze Pinecone.
